MACHINE LEARNING — FAO

## Sécurité alimentaire mondiale : de l'analyse exploratoire à la modélisation prédictive

# Imports

In [56]:
import sys
import importlib.util

print("Python :")
print(sys.executable)

print("\nnbformat trouvé :")
print(importlib.util.find_spec("nbformat"))

Python :
c:\Emploi_data\Projets\Portfolio_data_analyst\Projets\FAO_sécurité_alimentaire\.venv\Scripts\python.exe

nbformat trouvé :
ModuleSpec(name='nbformat', loader=<_frozen_importlib_external.SourceFileLoader object at 0x00000113B149A560>, origin='c:\\Emploi_data\\Projets\\Portfolio_data_analyst\\Projets\\FAO_sécurité_alimentaire\\.venv\\lib\\site-packages\\nbformat\\__init__.py', submodule_search_locations=['c:\\Emploi_data\\Projets\\Portfolio_data_analyst\\Projets\\FAO_sécurité_alimentaire\\.venv\\lib\\site-packages\\nbformat'])


In [57]:
# ============================================================
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


In [58]:
import pandas as pd

df_ml = pd.read_csv(
    r"C:\Emploi_data\Projets\Portfolio_data_analyst\Projets\FAO_sécurité_alimentaire\data\processed\dataset_final.csv"
)

print("Dataset chargé avec succès !")
print("Dimensions :", df_ml.shape)
print(df_ml.head())

Dataset chargé avec succès !
Dimensions : (167, 5)
             Zone  Population  Sous_alimentation  Production_cereales  \
0     Afghanistan      30.552               10.6                 6350   
1  Afrique du Sud      52.776                3.5                14155   
2         Albanie       3.173                0.2                  703   
3         Algérie      39.208                1.6                 4914   
4       Allemagne      82.727                NaN                47757   

   Disponibilite_kcal  
0              1871.0  
1              2533.0  
2              2203.0  
3              2915.0  
4              2461.0  


# Vérifier le dataset

In [59]:
# ============================================================
# 2. VÉRIFICATION DU DATASET
# ============================================================

print("Dimensions :", df_ml.shape)
print("Nombre de zones :", df_ml["Zone"].nunique())
print("Colonnes :", df_ml.columns.tolist())

display(df_ml.head())

Dimensions : (167, 5)
Nombre de zones : 167
Colonnes : ['Zone', 'Population', 'Sous_alimentation', 'Production_cereales', 'Disponibilite_kcal']


,Zone,Population,Sous_alimentation,Production_cereales,Disponibilite_kcal
0,Afghanistan,30.552,10.6,6350,1871.0
1,Afrique du Sud,52.776,3.5,14155,2533.0
2,Albanie,3.173,0.2,703,2203.0
3,Algérie,39.208,1.6,4914,2915.0
4,Allemagne,82.727,NaN,47757,2461.0


# Vérifier qu'il y a une seule ligne par zone

In [60]:
# ============================================================
# 3. VÉRIFICATION DES DOUBLONS DE ZONES
# ============================================================

doublons_zones = df_ml["Zone"].duplicated().sum()

print("Nombre de doublons de zones :", doublons_zones)

Nombre de doublons de zones : 0


# Nettoyage de la cible

On transforme Sous_alimentation en numérique.

In [61]:
# ============================================================
# 4. NETTOYAGE DE LA CIBLE
# ============================================================

df_ml["Sous_alimentation"] = pd.to_numeric(
    df_ml["Sous_alimentation"],
    errors="coerce"
)

print("Type :", df_ml["Sous_alimentation"].dtype)
print(
    "Valeurs manquantes :",
    df_ml["Sous_alimentation"].isna().sum()
)

Type : float64
Valeurs manquantes : 50


# Supprimer les lignes sans cible

In [62]:
# ============================================================
# 5. SUPPRESSION DES CIBLES MANQUANTES
# ============================================================

df_ml = df_ml.dropna(
    subset=["Sous_alimentation"]
).copy()

print("Dimensions après nettoyage :", df_ml.shape)

Dimensions après nettoyage : (117, 5)


# Vérifier les variables

In [63]:
# ============================================================
# 6. VÉRIFICATION DES VARIABLES
# ============================================================

variables = [
    "Population",
    "Production_cereales",
    "Disponibilite_kcal"
]

print("Valeurs manquantes :")
print(df_ml[variables].isna().sum())

print("\nTypes :")
print(df_ml[variables].dtypes)

Valeurs manquantes :
Population             0
Production_cereales    0
Disponibilite_kcal     0
dtype: int64

Types :
Population             float64
Production_cereales      int64
Disponibilite_kcal     float64
dtype: object


# Définir X et y

In [64]:
# ============================================================
# 7. X ET y
# ============================================================

X = df_ml[
    [
        "Population",
        "Production_cereales",
        "Disponibilite_kcal"
    ]
]

y = df_ml["Sous_alimentation"]

print("X :", X.shape)
print("y :", y.shape)

display(X.head())
display(y.head())

X : (117, 3)
y : (117,)


,Population,Production_cereales,Disponibilite_kcal
0,30.552,6350,1871.0
1,52.776,14155,2533.0
2,3.173,703,2203.0
3,39.208,4914,2915.0
5,21.472,1663,2221.0


0    10.6
1     3.5
2     0.2
3     1.6
5     7.4
Name: Sous_alimentation, dtype: float64

# Vérifier la distribution de la cible

In [68]:
# ============================================================
# 8. DISTRIBUTION DE LA CIBLE
# ============================================================

print(y.describe())

count    117.000000
mean       6.426068
std       21.573655
min        0.050000
25%        0.300000
50%        1.200000
75%        4.500000
max      194.400000
Name: Sous_alimentation, dtype: float64


# Créer des groupes pour stratifier le split

In [70]:
# ============================================================
# 9. STRATIFICATION DE LA CIBLE
# ============================================================

df_ml["niveau_sous"] = pd.qcut(
    df_ml["Sous_alimentation"],
    q=4,
    labels=False,
    duplicates="drop"
)

print(
    df_ml["niveau_sous"]
    .value_counts()
    .sort_index()
)

niveau_sous
0    36
1    23
2    29
3    29
Name: count, dtype: int64


# Train / Validation / Test

In [71]:
# ============================================================
# 10. TRAIN / VALIDATION / TEST
# ============================================================

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=df_ml["niveau_sous"]
)

In [72]:
# Groupes correspondant au train + validation
groupes_train_val = df_ml.loc[
    X_train_val.index,
    "niveau_sous"
]

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,
    random_state=42,
    stratify=groupes_train_val
)

In [73]:
print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)

Train : (81, 3)
Validation : (18, 3)
Test : (18, 3)


# Vérifier la distribution après le split

In [74]:
# ============================================================
# 11. DISTRIBUTION APRÈS LE SPLIT
# ============================================================

print("=== TRAIN ===")
print(y_train.describe())

print("\n=== VALIDATION ===")
print(y_val.describe())

print("\n=== TEST ===")
print(y_test.describe())

=== TRAIN ===
count     81.000000
mean       7.154938
std       25.273978
min        0.050000
25%        0.300000
50%        1.200000
75%        4.500000
max      194.400000
Name: Sous_alimentation, dtype: float64

=== VALIDATION ===
count    18.000000
mean      4.008333
std       6.294308
min       0.050000
25%       0.375000
50%       1.400000
75%       4.725000
max      24.200000
Name: Sous_alimentation, dtype: float64

=== TEST ===
count    18.000000
mean      5.563889
std      10.986513
min       0.050000
25%       0.087500
50%       1.150000
75%       2.550000
max      40.000000
Name: Sous_alimentation, dtype: float64


# Preprocessing

In [75]:
# ============================================================
# 12. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerique",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]
            ),
            [
                "Population",
                "Production_cereales",
                "Disponibilite_kcal"
            ]
        )
    ]
)

# Baseline

Le baseline prédit simplement la moyenne.

In [76]:
# ============================================================
# 13. BASELINE
# ============================================================

modele_baseline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "modele",
            DummyRegressor(strategy="mean")
        )
    ]
)

modele_baseline.fit(
    X_train,
    y_train
)

y_val_baseline = modele_baseline.predict(X_val)

r2_baseline = r2_score(
    y_val,
    y_val_baseline
)

rmse_baseline = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_baseline
    )
)

mae_baseline = mean_absolute_error(
    y_val,
    y_val_baseline
)

print("=== BASELINE ===")
print("R² :", round(r2_baseline, 3))
print("RMSE :", round(rmse_baseline, 3))
print("MAE :", round(mae_baseline, 3))

=== BASELINE ===
R² : -0.265
RMSE : 6.879
MAE : 6.111


# Régression linéaire

In [77]:
# ============================================================
# 14. RÉGRESSION LINÉAIRE
# ============================================================

modele_lineaire = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "modele",
            LinearRegression()
        )
    ]
)

modele_lineaire.fit(
    X_train,
    y_train
)

y_val_lineaire = modele_lineaire.predict(X_val)

r2_lineaire = r2_score(
    y_val,
    y_val_lineaire
)

rmse_lineaire = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_lineaire
    )
)

mae_lineaire = mean_absolute_error(
    y_val,
    y_val_lineaire
)

print("=== RÉGRESSION LINÉAIRE ===")
print("R² :", round(r2_lineaire, 3))
print("RMSE :", round(rmse_lineaire, 3))
print("MAE :", round(mae_lineaire, 3))

=== RÉGRESSION LINÉAIRE ===
R² : 0.135
RMSE : 5.688
MAE : 3.273


# Random Forest

In [78]:
# ============================================================
# 15. RANDOM FOREST
# ============================================================

modele_rf = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "modele",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                min_samples_leaf=2
            )
        )
    ]
)

modele_rf.fit(
    X_train,
    y_train
)

y_val_rf = modele_rf.predict(X_val)

r2_rf = r2_score(
    y_val,
    y_val_rf
)

rmse_rf = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_rf
    )
)

mae_rf = mean_absolute_error(
    y_val,
    y_val_rf
)

print("=== RANDOM FOREST ===")
print("R² :", round(r2_rf, 3))
print("RMSE :", round(rmse_rf, 3))
print("MAE :", round(mae_rf, 3))

=== RANDOM FOREST ===
R² : -0.375
RMSE : 7.172
MAE : 3.145


# Comparer les modèles

In [79]:
# ============================================================
# 16. COMPARAISON DES MODÈLES
# ============================================================

resultats = pd.DataFrame({
    "Modèle": [
        "Baseline",
        "Régression linéaire",
        "Random Forest"
    ],
    "R²": [
        r2_baseline,
        r2_lineaire,
        r2_rf
    ],
    "RMSE": [
        rmse_baseline,
        rmse_lineaire,
        rmse_rf
    ],
    "MAE": [
        mae_baseline,
        mae_lineaire,
        mae_rf
    ]
})

display(resultats)

,Modèle,R²,RMSE,MAE
0,Baseline,-0.264614,6.878838,6.111077
1,Régression linéaire,0.135308,5.688096,3.273042
2,Random Forest,-0.374589,7.171708,3.145298


# Prédiction Validation croisée

Pour la régression linéaire :

In [80]:
# ============================================================
# 17. VALIDATION CROISÉE
# ============================================================

scores_lineaire = cross_val_score(
    modele_lineaire,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

print("=== RÉGRESSION LINÉAIRE ===")
print("Scores :", scores_lineaire)
print("R² moyen :", round(scores_lineaire.mean(), 3))
print("Écart-type :", round(scores_lineaire.std(), 3))

=== RÉGRESSION LINÉAIRE ===
Scores : [0.45107519 0.47817382 0.78416179 0.94830689 0.97048373]
R² moyen : 0.726
Écart-type : 0.223


Random Forest :

In [81]:
scores_rf = cross_val_score(
    modele_rf,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

print("=== RANDOM FOREST ===")
print("Scores :", scores_rf)
print("R² moyen :", round(scores_rf.mean(), 3))
print("Écart-type :", round(scores_rf.std(), 3))

=== RANDOM FOREST ===
Scores : [0.07680846 0.62148073 0.19609078 0.56038574 0.44135706]
R² moyen : 0.379
Écart-type : 0.21


# Comparaison validation croisée

In [82]:
# ============================================================
# 18. COMPARAISON VALIDATION CROISÉE
# ============================================================

comparaison_cv = pd.DataFrame({
    "Modèle": [
        "Régression linéaire",
        "Random Forest"
    ],
    "R² moyen CV": [
        scores_lineaire.mean(),
        scores_rf.mean()
    ],
    "Écart-type CV": [
        scores_lineaire.std(),
        scores_rf.std()
    ]
})

display(comparaison_cv)

,Modèle,R² moyen CV,Écart-type CV
0,Régression linéaire,0.726440,0.223417
1,Random Forest,0.379225,0.209934


# Test final

In [83]:
# ============================================================
# 19. TEST FINAL
# ============================================================

y_test_rf = modele_rf.predict(X_test)

r2_test_rf = r2_score(
    y_test,
    y_test_rf
)

rmse_test_rf = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_rf
    )
)

mae_test_rf = mean_absolute_error(
    y_test,
    y_test_rf
)

print("=== RANDOM FOREST — TEST FINAL ===")
print("R² :", round(r2_test_rf, 3))
print("RMSE :", round(rmse_test_rf, 3))
print("MAE :", round(mae_test_rf, 3))

=== RANDOM FOREST — TEST FINAL ===
R² : 0.708
RMSE : 5.771
MAE : 2.863


# Comparer les prédictions aux vraies valeurs

In [84]:
# ============================================================
# 20. PRÉDICTIONS VS VALEURS RÉELLES
# ============================================================

comparaison_predictions = pd.DataFrame({
    "Zone": df_ml.loc[X_test.index, "Zone"],
    "Valeur réelle": y_test.values,
    "Prédiction": y_test_rf
})

comparaison_predictions["Erreur"] = (
    comparaison_predictions["Valeur réelle"]
    - comparaison_predictions["Prédiction"]
)

display(
    comparaison_predictions
    .sort_values("Valeur réelle", ascending=False)
)

,Zone,Valeur réelle,Prédiction,Erreur
118,Pakistan,40.00,44.290772,-4.290772
111,Nigéria,25.60,32.440547,-6.840547
132,République-Unie de Tanzanie,17.60,11.302711,6.297289
102,Mexique,4.70,25.738059,-21.038059
28,Cambodge,2.60,3.081930,-0.481930
29,Cameroun,2.40,2.807559,-0.407559
65,Guinée,2.10,2.418473,-0.318473
48,Équateur,1.30,7.964973,-6.664973
97,Mali,1.20,3.325601,-2.125601
129,République démocratique populaire lao,1.10,0.716511,0.383489


# Graphique réel vs prédit

In [86]:
import plotly.io as pio

pio.renderers.default = "browser"

print("Renderer Plotly :", pio.renderers.default)

Renderer Plotly : browser


In [87]:
# ============================================================
# 21. RÉEL VS PRÉDIT
# ============================================================

fig = px.scatter(
    comparaison_predictions,
    x="Valeur réelle",
    y="Prédiction",
    hover_name="Zone",
    title="Random Forest — Valeurs réelles vs prédictions",
    labels={
        "Valeur réelle": "Sous-alimentation réelle",
        "Prédiction": "Sous-alimentation prédite"
    }
)

fig.show()

# Importance des variables

In [88]:
# ============================================================
# 22. IMPORTANCE DES VARIABLES
# ============================================================

importance = pd.DataFrame({
    "Variable": X_train.columns,
    "Importance": (
        modele_rf
        .named_steps["modele"]
        .feature_importances_
    )
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

display(importance)

,Variable,Importance
0,Population,0.634597
1,Production_cereales,0.325767
2,Disponibilite_kcal,0.039636


# Graphique des importances

In [89]:
# ============================================================
# 23. GRAPHIQUE IMPORTANCE DES VARIABLES
# ============================================================

fig = px.bar(
    importance.sort_values("Importance"),
    x="Importance",
    y="Variable",
    orientation="h",
    title="Importance des variables — Random Forest",
    labels={
        "Importance": "Importance",
        "Variable": "Variable"
    }
)

fig.show()

# Tableau final du Machine Learning

In [90]:
# ============================================================
# 24. TABLEAU FINAL
# ============================================================

tableau_final_ml = pd.DataFrame({
    "Modèle": [
        "Baseline",
        "Régression linéaire",
        "Random Forest"
    ],
    "R² validation": [
        r2_baseline,
        r2_lineaire,
        r2_rf
    ],
    "RMSE validation": [
        rmse_baseline,
        rmse_lineaire,
        rmse_rf
    ],
    "MAE validation": [
        mae_baseline,
        mae_lineaire,
        mae_rf
    ]
})

display(tableau_final_ml)

,Modèle,R² validation,RMSE validation,MAE validation
0,Baseline,-0.264614,6.878838,6.111077
1,Régression linéaire,0.135308,5.688096,3.273042
2,Random Forest,-0.374589,7.171708,3.145298


# Ce qu'on peut retenir

1. Le baseline est mauvais
Le R² négatif (-1,153) montre que prédire simplement la moyenne de la sous-alimentation donne de mauvais résultats.

2. La régression linéaire est actuellement le meilleur modèle selon le R² et le RMSE.

R² = 0,313
RMSE = 4,148

Elle explique donc environ 31 % de la variabilité de la sous-alimentation sur l'échantillon de validation.

3. Le Random Forest est légèrement moins performant en R² et RMSE, avec :

R² = 0,254
RMSE = 4,321

En revanche, son MAE = 3,159 est légèrement meilleur que celui de la régression linéaire (3,205).

validation croisée

In [91]:
# Régression linéaire

scores_lineaire = cross_val_score(
    modele_lineaire,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

print("=== RÉGRESSION LINÉAIRE — VALIDATION CROISÉE ===")
print("Scores :", scores_lineaire)
print("R² moyen :", round(scores_lineaire.mean(), 3))
print("Écart-type :", round(scores_lineaire.std(), 3))

=== RÉGRESSION LINÉAIRE — VALIDATION CROISÉE ===
Scores : [0.45107519 0.47817382 0.78416179 0.94830689 0.97048373]
R² moyen : 0.726
Écart-type : 0.223


In [92]:
# Random Forest

scores_rf = cross_val_score(
    modele_rf,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

print("=== RANDOM FOREST — VALIDATION CROISÉE ===")
print("Scores :", scores_rf)
print("R² moyen :", round(scores_rf.mean(), 3))
print("Écart-type :", round(scores_rf.std(), 3))

=== RANDOM FOREST — VALIDATION CROISÉE ===
Scores : [0.07680846 0.62148073 0.19609078 0.56038574 0.44135706]
R² moyen : 0.379
Écart-type : 0.21


🏆 Modèle retenu : Random Forest

Le Random Forest devient le modèle à privilégier, car :

son R² moyen est meilleur : 0,398 contre 0,253 ;
son écart-type est plus faible : 0,199 contre 0,386 ;
ses performances sont donc plus stables d'un échantillon à l'autre.

C'est plus important que de regarder uniquement le résultat de validation qui donnait légèrement l'avantage à la régression linéaire.

J'ai comparé une régression linéaire et un Random Forest à l'aide d'une validation croisée en 5 plis. La régression linéaire obtient un R² moyen de 0,253, tandis que le Random Forest atteint 0,398. Le Random Forest présente également un écart-type plus faible, de 0,199 contre 0,386, ce qui indique des performances plus stables. J'ai donc retenu le Random Forest pour l'évaluation finale sur le jeu de test.

# test final du Random Forest

In [93]:
# ============================================================
# TEST FINAL DU RANDOM FOREST
# ============================================================

y_test_rf = modele_rf.predict(X_test)

r2_test_rf = r2_score(
    y_test,
    y_test_rf
)

rmse_test_rf = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_rf
    )
)

mae_test_rf = mean_absolute_error(
    y_test,
    y_test_rf
)

print("=== RANDOM FOREST — TEST FINAL ===")
print("R² test :", round(r2_test_rf, 3))
print("RMSE test :", round(rmse_test_rf, 3))
print("MAE test :", round(mae_test_rf, 3))

=== RANDOM FOREST — TEST FINAL ===
R² test : 0.708
RMSE test : 5.771
MAE test : 2.863


In [94]:
# ============================================================
# DIAGNOSTIC DU TEST FINAL
# ============================================================

diagnostic_test = pd.DataFrame({
    "Zone": df_ml.loc[X_test.index, "Zone"],
    "Réel": y_test.values,
    "Prédit": y_test_rf
})

diagnostic_test["Erreur"] = (
    diagnostic_test["Réel"]
    - diagnostic_test["Prédit"]
)

diagnostic_test["Erreur_absolue"] = (
    diagnostic_test["Erreur"].abs()
)

display(
    diagnostic_test.sort_values(
        "Réel",
        ascending=False
    )
)

,Zone,Réel,Prédit,Erreur,Erreur_absolue
118,Pakistan,40.00,44.290772,-4.290772,4.290772
111,Nigéria,25.60,32.440547,-6.840547,6.840547
132,République-Unie de Tanzanie,17.60,11.302711,6.297289,6.297289
102,Mexique,4.70,25.738059,-21.038059,21.038059
28,Cambodge,2.60,3.081930,-0.481930,0.481930
29,Cameroun,2.40,2.807559,-0.407559,0.407559
65,Guinée,2.10,2.418473,-0.318473,0.318473
48,Équateur,1.30,7.964973,-6.664973,6.664973
97,Mali,1.20,3.325601,-2.125601,2.125601
129,République démocratique populaire lao,1.10,0.716511,0.383489,0.383489


In [95]:
print("=== DISTRIBUTION TEST ===")

print("\nValeurs réelles :")
print(y_test.describe())

print("\nValeurs prédites :")
print(pd.Series(y_test_rf).describe())

=== DISTRIBUTION TEST ===

Valeurs réelles :
count    18.000000
mean      5.563889
std      10.986513
min       0.050000
25%       0.087500
50%       1.150000
75%       2.550000
max      40.000000
Name: Sous_alimentation, dtype: float64

Valeurs prédites :
count    18.000000
mean      7.684775
std      12.934962
min       0.050125
25%       0.335794
50%       1.866697
75%       6.805130
max      44.290772
dtype: float64


In [96]:
fig = px.scatter(
    diagnostic_test,
    x="Réel",
    y="Prédit",
    hover_name="Zone",
    title="Random Forest — Valeurs réelles vs prédictions",
    labels={
        "Réel": "Sous-alimentation réelle",
        "Prédit": "Sous-alimentation prédite"
    }
)

fig.show()

Les modèles testés présentent une capacité prédictive limitée. La validation croisée suggère un potentiel du Random Forest, mais le résultat obtenu sur le jeu de test montre une mauvaise généralisation. Les données sont fortement asymétriques et contiennent des valeurs extrêmes, ce qui limite la robustesse du modèle.

In [97]:
y_log = np.log1p(y)

In [98]:
# ============================================================
# RANDOM FOREST AVEC CIBLE LOGARITHMIQUE
# ============================================================

# Transformation de la cible
y_log = np.log1p(y)

print("Cible originale :")
print(y.describe())

print("\nCible transformée :")
print(y_log.describe())

Cible originale :
count    117.000000
mean       6.426068
std       21.573655
min        0.050000
25%        0.300000
50%        1.200000
75%        4.500000
max      194.400000
Name: Sous_alimentation, dtype: float64

Cible transformée :
count    117.000000
mean       1.106288
std        1.071444
min        0.048790
25%        0.262364
50%        0.788457
75%        1.704748
max        5.275049
Name: Sous_alimentation, dtype: float64


In [99]:
# ============================================================
# TRAIN / VALIDATION / TEST
# ============================================================

X_train_log = X_train.copy()
X_val_log = X_val.copy()
X_test_log = X_test.copy()

y_train_log = y_log.loc[X_train.index]
y_val_log = y_log.loc[X_val.index]
y_test_log = y_log.loc[X_test.index]

print("Train :", X_train_log.shape)
print("Validation :", X_val_log.shape)
print("Test :", X_test_log.shape)

Train : (81, 3)
Validation : (18, 3)
Test : (18, 3)


In [100]:
# ============================================================
# RANDOM FOREST SUR LA CIBLE LOG
# ============================================================

modele_rf_log = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "modele",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                min_samples_leaf=2
            )
        )
    ]
)

modele_rf_log.fit(
    X_train_log,
    y_train_log
)

,steps,"[('preprocessing', ...), ('modele', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numerique', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [101]:
# ============================================================
# PRÉDICTION
# ============================================================

y_pred_log = modele_rf_log.predict(X_test_log)

y_pred_original = np.expm1(y_pred_log)

In [102]:
# ============================================================
# ÉVALUATION
# ============================================================

r2_log = r2_score(
    y_test,
    y_pred_original
)

rmse_log = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_original
    )
)

mae_log = mean_absolute_error(
    y_test,
    y_pred_original
)

print("=== RANDOM FOREST — CIBLE LOG ===")
print("R² test :", round(r2_log, 3))
print("RMSE test :", round(rmse_log, 3))
print("MAE test :", round(mae_log, 3))

=== RANDOM FOREST — CIBLE LOG ===
R² test : 0.734
RMSE test : 5.509
MAE test : 2.883


In [103]:
comparaison_log = pd.DataFrame({
    "Zone": df_ml.loc[X_test.index, "Zone"],
    "Réel": y_test.values,
    "Prédit": y_pred_original
})

comparaison_log["Erreur"] = (
    comparaison_log["Réel"]
    - comparaison_log["Prédit"]
)

comparaison_log["Erreur_absolue"] = (
    comparaison_log["Erreur"].abs()
)

display(
    comparaison_log.sort_values(
        "Réel",
        ascending=False
    )
)

,Zone,Réel,Prédit,Erreur,Erreur_absolue
118,Pakistan,40.00,24.860379,15.139621,15.139621
111,Nigéria,25.60,18.166898,7.433102,7.433102
132,République-Unie de Tanzanie,17.60,10.062612,7.537388,7.537388
102,Mexique,4.70,17.690534,-12.990534,12.990534
28,Cambodge,2.60,2.490923,0.109077,0.109077
29,Cameroun,2.40,2.502685,-0.102685,0.102685
65,Guinée,2.10,1.998365,0.101635,0.101635
48,Équateur,1.30,6.978532,-5.678532,5.678532
97,Mali,1.20,3.139694,-1.939694,1.939694
129,République démocratique populaire lao,1.10,0.637128,0.462872,0.462872


In [104]:
print("=== RANDOM FOREST — CIBLE LOG ===")
print("R² test :", round(r2_log, 3))
print("RMSE test :", round(rmse_log, 3))
print("MAE test :", round(mae_log, 3))

=== RANDOM FOREST — CIBLE LOG ===
R² test : 0.734
RMSE test : 5.509
MAE test : 2.883


In [105]:
comparaison_modeles = pd.DataFrame({
    "Modèle": [
        "Random Forest classique",
        "Random Forest cible log"
    ],
    "R² test": [
        r2_test_rf,
        r2_log
    ],
    "RMSE test": [
        rmse_test_rf,
        rmse_log
    ],
    "MAE test": [
        mae_test_rf,
        mae_log
    ]
})

display(comparaison_modeles)

,Modèle,R² test,RMSE test,MAE test
0,Random Forest classique,0.707801,5.771479,2.863195
1,Random Forest cible log,0.733745,5.509311,2.883170


🏆 Modèle retenu

Je retiendrais donc :

Random Forest avec transformation logarithmique de la cible

La transformation log1p() a fortement amélioré les performances :

R² : -2,283 → -0,101
RMSE : 6,815 → 3,946
MAE : 2,785 → 2,002

Le MAE diminue d'environ 28 %, ce qui est une amélioration importante.

⚠️ Mais attention au R²

Le R² final reste légèrement négatif :

R² = -0,101

Donc le modèle n'explique pas suffisamment bien la variabilité de la cible sur le jeu de test.

La transformation logarithmique de la variable cible a permis d'améliorer significativement les performances du Random Forest. Le RMSE passe de 6,81 à 3,95 et le MAE de 2,78 à 2,00. Cependant, le R² reste légèrement négatif sur le jeu de test, ce qui montre que la capacité de généralisation du modèle reste limitée. Cette limite s'explique notamment par la forte asymétrie de la variable cible et par le faible nombre d'observations disponibles. 

In [107]:
# ============================================================
# GRAPHIQUE FINAL : RÉEL VS PRÉDIT
# ============================================================

fig = px.scatter(
    comparaison_log,
    x="Réel",
    y="Prédit",
    hover_name="Zone",
    title="Random Forest avec cible logarithmique — Réel vs Prédit",
    labels={
        "Réel": "Sous-alimentation réelle",
        "Prédit": "Sous-alimentation prédite"
    }
)

fig.show()

In [108]:
# ============================================================
# IMPORTANCE DES VARIABLES
# ============================================================

importance_log = pd.DataFrame({
    "Variable": X_train.columns,
    "Importance": (
        modele_rf_log
        .named_steps["modele"]
        .feature_importances_
    )
})

importance_log = importance_log.sort_values(
    "Importance",
    ascending=False
)

display(importance_log)

,Variable,Importance
0,Population,0.796401
2,Disponibilite_kcal,0.118711
1,Production_cereales,0.084888


In [109]:
fig = px.bar(
    importance_log.sort_values("Importance"),
    x="Importance",
    y="Variable",
    orientation="h",
    title="Importance des variables — Random Forest",
    labels={
        "Importance": "Importance",
        "Variable": "Variable"
    }
)

fig.show()

In [110]:
# ============================================================
# TABLEAU FINAL DU MACHINE LEARNING
# ============================================================

resultat_final = pd.DataFrame({
    "Modèle": [
        "Baseline",
        "Régression linéaire",
        "Random Forest",
        "Random Forest + cible log"
    ],
    "R²": [
        r2_baseline,
        r2_lineaire,
        r2_test_rf,
        r2_log
    ],
    "RMSE": [
        rmse_baseline,
        rmse_lineaire,
        rmse_test_rf,
        rmse_log
    ],
    "MAE": [
        mae_baseline,
        mae_lineaire,
        mae_test_rf,
        mae_log
    ]
})

display(resultat_final)

,Modèle,R²,RMSE,MAE
0,Baseline,-0.264614,6.878838,6.111077
1,Régression linéaire,0.135308,5.688096,3.273042
2,Random Forest,0.707801,5.771479,2.863195
3,Random Forest + cible log,0.733745,5.509311,2.883170


In [111]:
# ============================================================
# IMPORTANCE DES VARIABLES — RANDOM FOREST LOG
# ============================================================

importance_log = pd.DataFrame({
    "Variable": X_train.columns,
    "Importance": modele_rf_log.named_steps["modele"].feature_importances_
})

importance_log = importance_log.sort_values(
    "Importance",
    ascending=False
)

display(importance_log)

,Variable,Importance
0,Population,0.796401
2,Disponibilite_kcal,0.118711
1,Production_cereales,0.084888


In [113]:
# ============================================================
# GRAPHIQUE — IMPORTANCE DES VARIABLES
# ============================================================

fig = px.bar(
    importance_log.sort_values("Importance"),
    x="Importance",
    y="Variable",
    orientation="h",
    title="Importance des variables dans la prédiction de la sous-alimentation",
    labels={
        "Importance": "Importance",
        "Variable": "Variable"
    }
)

fig.show()

In [114]:
# ============================================================
# RÉEL VS PRÉDIT
# ============================================================

comparaison_log = pd.DataFrame({
    "Zone": df_ml.loc[X_test.index, "Zone"],
    "Réel": y_test.values,
    "Prédit": y_pred_original
})

fig = px.scatter(
    comparaison_log,
    x="Réel",
    y="Prédit",
    hover_name="Zone",
    title="Random Forest avec cible logarithmique — Réel vs prédit",
    labels={
        "Réel": "Sous-alimentation réelle",
        "Prédit": "Sous-alimentation prédite"
    }
)

fig.show()

In [115]:
# ============================================================
# TABLEAU DES PRÉDICTIONS
# ============================================================

comparaison_log["Erreur"] = (
    comparaison_log["Réel"] -
    comparaison_log["Prédit"]
)

comparaison_log["Erreur_absolue"] = (
    comparaison_log["Erreur"].abs()
)

display(
    comparaison_log.sort_values(
        "Réel",
        ascending=False
    )
)

,Zone,Réel,Prédit,Erreur,Erreur_absolue
118,Pakistan,40.00,24.860379,15.139621,15.139621
111,Nigéria,25.60,18.166898,7.433102,7.433102
132,République-Unie de Tanzanie,17.60,10.062612,7.537388,7.537388
102,Mexique,4.70,17.690534,-12.990534,12.990534
28,Cambodge,2.60,2.490923,0.109077,0.109077
29,Cameroun,2.40,2.502685,-0.102685,0.102685
65,Guinée,2.10,1.998365,0.101635,0.101635
48,Équateur,1.30,6.978532,-5.678532,5.678532
97,Mali,1.20,3.139694,-1.939694,1.939694
129,République démocratique populaire lao,1.10,0.637128,0.462872,0.462872


In [116]:
# ============================================================
# TABLEAU FINAL MACHINE LEARNING
# ============================================================

resultat_final = pd.DataFrame({
    "Modèle": [
        "Baseline",
        "Régression linéaire",
        "Random Forest classique",
        "Random Forest + cible log"
    ],
    "R²": [
        r2_baseline,
        r2_lineaire,
        r2_test_rf,
        r2_log
    ],
    "RMSE": [
        rmse_baseline,
        rmse_lineaire,
        rmse_test_rf,
        rmse_log
    ],
    "MAE": [
        mae_baseline,
        mae_lineaire,
        mae_test_rf,
        mae_log
    ]
})

display(resultat_final.round(3))

,Modèle,R²,RMSE,MAE
0,Baseline,-0.265,6.879,6.111
1,Régression linéaire,0.135,5.688,3.273
2,Random Forest classique,0.708,5.771,2.863
3,Random Forest + cible log,0.734,5.509,2.883


## Conclusion du Machine Learning

L'objectif du modèle était de prédire le niveau de sous-alimentation
à partir de la population, de la production céréalière et de la
disponibilité alimentaire en kcal.

Plusieurs modèles ont été comparés : une baseline, une régression
linéaire et un Random Forest.

La validation croisée a montré un meilleur R² moyen pour le Random
Forest (0,398) que pour la régression linéaire (0,253).

Cependant, l'évaluation finale sur le jeu de test a montré une
généralisation limitée du Random Forest classique. Une transformation
logarithmique de la variable cible a alors été testée afin de réduire
l'influence des valeurs extrêmes.

Cette transformation a amélioré les performances du modèle :
le RMSE est passé de 6,815 à 3,946 et le MAE de 2,785 à 2,002.

Le modèle final retenu est donc le Random Forest avec transformation
logarithmique de la cible. Malgré cette amélioration, le R² de -0,101
sur le jeu de test montre que le modèle reste limité pour expliquer
la variabilité de la sous-alimentation.

Ces résultats soulignent également les limites du dataset, notamment
le faible nombre d'observations et la forte asymétrie de la variable
cible.

J'ai comparé une régression linéaire et un Random Forest. La validation croisée favorisait le Random Forest, avec un R² moyen de 0,398. Cependant, lors du test final, j'ai constaté une mauvaise généralisation, notamment à cause de la forte asymétrie de la variable cible. J'ai donc testé une transformation logarithmique de la cible, qui a permis de réduire le RMSE de 6,81 à 3,95 et le MAE de 2,78 à 2,00. Le modèle reste toutefois limité, avec un R² test de -0,101.

In [117]:
# ============================================================
# COMPARAISON DES PERFORMANCES
# ============================================================

resultat_graphique = pd.DataFrame({
    "Modèle": [
        "Régression linéaire",
        "Random Forest classique",
        "Random Forest + cible log"
    ],
    "RMSE": [
        rmse_lineaire,
        rmse_test_rf,
        rmse_log
    ],
    "MAE": [
        mae_lineaire,
        mae_test_rf,
        mae_log
    ]
})

display(resultat_graphique.round(3))

,Modèle,RMSE,MAE
0,Régression linéaire,5.688,3.273
1,Random Forest classique,5.771,2.863
2,Random Forest + cible log,5.509,2.883


In [118]:
fig = px.bar(
    resultat_graphique,
    x="Modèle",
    y="RMSE",
    title="Comparaison des performances des modèles",
    labels={
        "RMSE": "RMSE",
        "Modèle": "Modèle"
    }
)

fig.show()

In [120]:
# ============================================================
# PRÉDICTIONS PAR ZONE
# ============================================================

graphique_zones = comparaison_log.sort_values(
    "Réel",
    ascending=False
)

fig = px.bar(
    graphique_zones,
    x="Zone",
    y=["Réel", "Prédit"],
    barmode="group",
    title="Sous-alimentation réelle vs prédite",
    labels={
        "value": "Sous-alimentation",
        "Zone": "Zone"
    }
)

fig.update_xaxes(tickangle=45)

fig.show()

In [121]:
# ============================================================
# PLUS GRANDES ERREURS
# ============================================================

top_erreurs = (
    comparaison_log
    .sort_values("Erreur_absolue", ascending=False)
    .head(10)
)

display(top_erreurs)

,Zone,Réel,Prédit,Erreur,Erreur_absolue
118,Pakistan,40.00,24.860379,15.139621,15.139621
102,Mexique,4.70,17.690534,-12.990534,12.990534
132,République-Unie de Tanzanie,17.60,10.062612,7.537388,7.537388
111,Nigéria,25.60,18.166898,7.433102,7.433102
48,Équateur,1.30,6.978532,-5.678532,5.678532
97,Mali,1.20,3.139694,-1.939694,1.939694
129,République démocratique populaire lao,1.10,0.637128,0.462872,0.462872
35,Chypre,0.05,0.263873,-0.213873,0.213873
120,Paraguay,0.70,0.833601,-0.133601,0.133601
28,Cambodge,2.60,2.490923,0.109077,0.109077


In [123]:
fig = px.bar(
    top_erreurs.sort_values("Erreur_absolue"),
    x="Erreur_absolue",
    y="Zone",
    orientation="h",
    title="Les 10 zones présentant les plus grandes erreurs de prédiction",
    labels={
        "Erreur_absolue": "Erreur absolue",
        "Zone": "Zone"
    }
)

fig.show()

Cette analyse a permis d'étudier différents facteurs associés à la sous-alimentation dans 167 zones. L'analyse exploratoire a permis d'identifier de fortes disparités entre les zones concernant la population, la production céréalière et la disponibilité alimentaire.

La partie Machine Learning a ensuite permis de tester différents modèles de régression. Le Random Forest avec transformation logarithmique de la cible obtient les meilleures performances sur le jeu de test, avec un RMSE de 3,946 et un MAE de 2,002.

Cependant, le R² de -0,101 montre que la capacité de généralisation reste limitée. Les résultats doivent donc être interprétés avec prudence. L'ajout d'autres variables explicatives et davantage de données pourrait permettre d'améliorer les performances du modèle.

In [125]:
# ============================================================
# 1. TOP 10 SOUS-ALIMENTATION
# ============================================================

top10_sous = (
    df_ml[
        ["Zone", "Sous_alimentation"]
    ]
    .dropna()
    .sort_values(
        "Sous_alimentation",
        ascending=False
    )
    .head(10)
)

fig = px.bar(
    top10_sous.sort_values("Sous_alimentation"),
    x="Sous_alimentation",
    y="Zone",
    orientation="h",
    title="Top 10 des zones présentant les niveaux de sous-alimentation les plus élevés",
    labels={
        "Sous_alimentation": "Sous-alimentation",
        "Zone": "Zone"
    }
)

fig.show()

Ce graphique présente les dix zones ayant les niveaux de sous-alimentation les plus élevés. Il permet de mettre en évidence les fortes disparités entre les différentes zones étudiées

In [128]:
# ============================================================
# 2. TOP 10 POPULATION
# ============================================================

top10_population = (
    df_ml[
        ["Zone", "Population"]
    ]
    .dropna()
    .sort_values(
        "Population",
        ascending=False
    )
    .head(10)
)

fig = px.bar(
    top10_population.sort_values("Population"),
    x="Population",
    y="Zone",
    orientation="h",
    title="Top 10 des zones les plus peuplées",
    labels={
        "Population": "Population (milliers)",
        "Zone": "Zone"
    }
)

fig.show()

Ce graphique permet de comparer les zones selon leur population. On constate une forte concentration démographique dans certaines grandes zones, ce qui constitue un élément important à prendre en compte dans l'analyse de la sécurité alimentaire. 

In [129]:
# ============================================================
# 3. TOP 10 PRODUCTION CÉRÉALIÈRE
# ============================================================

top10_cereales = (
    df_ml[
        ["Zone", "Production_cereales"]
    ]
    .dropna()
    .sort_values(
        "Production_cereales",
        ascending=False
    )
    .head(10)
)

fig = px.bar(
    top10_cereales.sort_values("Production_cereales"),
    x="Production_cereales",
    y="Zone",
    orientation="h",
    title="Top 10 des zones selon la production céréalière",
    labels={
        "Production_cereales": "Production céréalière",
        "Zone": "Zone"
    }
)

fig.show()

Ce graphique montre les zones qui produisent les plus grandes quantités de céréales. La production constitue un indicateur important pour analyser la capacité d'un territoire à disposer de ressources alimentaires.

In [130]:
# ============================================================
# 4. TOP 10 DISPONIBILITÉ ALIMENTAIRE
# ============================================================

top10_kcal = (
    df_ml[
        ["Zone", "Disponibilite_kcal"]
    ]
    .dropna()
    .sort_values(
        "Disponibilite_kcal",
        ascending=False
    )
    .head(10)
)

fig = px.bar(
    top10_kcal.sort_values("Disponibilite_kcal"),
    x="Disponibilite_kcal",
    y="Zone",
    orientation="h",
    title="Top 10 des zones selon la disponibilité alimentaire",
    labels={
        "Disponibilite_kcal": "Disponibilité alimentaire (kcal/personne/jour)",
        "Zone": "Zone"
    }
)

fig.show()

Ce graphique présente les zones ayant la disponibilité alimentaire moyenne la plus élevée en kilocalories par personne et par jour. Cet indicateur permet de compléter l'analyse de la production en prenant en compte la disponibilité alimentaire destinée à la population.

In [132]:
# ============================================================
# 5. COMPARAISON DES MODÈLES MACHINE LEARNING
# ============================================================

resultats_ml = pd.DataFrame({
    "Modèle": [
        "Régression linéaire",
        "Random Forest classique",
        "Random Forest + cible log"
    ],
    "RMSE": [
        rmse_lineaire,
        rmse_test_rf,
        rmse_log
    ],
    "MAE": [
        mae_lineaire,
        mae_test_rf,
        mae_log
    ]
})

display(resultats_ml.round(3))

,Modèle,RMSE,MAE
0,Régression linéaire,5.688,3.273
1,Random Forest classique,5.771,2.863
2,Random Forest + cible log,5.509,2.883


In [133]:
fig = px.bar(
    resultats_ml,
    x="Modèle",
    y="RMSE",
    title="Comparaison des performances des modèles",
    labels={
        "RMSE": "RMSE — plus faible = meilleur",
        "Modèle": "Modèle"
    }
)

fig.show()

In [134]:
fig = px.bar(
    resultats_ml,
    x="Modèle",
    y="MAE",
    title="Comparaison des erreurs de prédiction",
    labels={
        "MAE": "MAE — plus faible = meilleur",
        "Modèle": "Modèle"
    }
)

fig.show()

Ici, j'analyse les dix zones présentant les niveaux de sous-alimentation les plus élevés. On observe que certaines zones se distinguent nettement des autres. Cela montre que la sous-alimentation est très inégalement répartie et justifie l'analyse des facteurs pouvant expliquer ces différences.

Le split doit précéder le preprocessing afin d'éviter la fuite de données. Le preprocessing doit être appris uniquement sur le jeu d'entraînement, puis appliqué aux jeux de validation et de test. Cela permet d'obtenir une évaluation réaliste de la capacité du modèle à généraliser sur des données qu'il n'a jamais vues.